# LoRA vs QLoRA Fine-Tuning of Qwen2.5-1.5B-Instruct on Agricultural Q&A

# **1. Environment Setup**

In [1]:
!pip uninstall -y torchao

!pip install -q \
transformers==4.56.1 \
peft==0.17.1 \
trl==0.21.0 \
accelerate \
bitsandbytes \
datasets \
evaluate \
bert_score \
sentencepiece

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 88.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.3 MB/s eta 0:00:00


In [2]:
import transformers
import peft
import trl
import torch

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)

Torch: 2.10.0+cu128
Transformers: 4.56.1
PEFT: 0.17.1
TRL: 0.21.0


## 2. GPU Verification

In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    print("bf16 supported:", torch.cuda.is_bf16_supported())
else:
    print("WARNING: no GPU detected. Enable a GPU accelerator in Kaggle notebook settings.")

CUDA available: True
Device: Tesla T4
VRAM (GB): 15.6
bf16 supported: True


## 3. Reproducibility

In [4]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

## 4. Dataset Loading

In [5]:
from datasets import load_dataset

raw = load_dataset("KisanVaani/agriculture-qa-english-only", split="train")
print(raw)
print(raw[0])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.97M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/22615 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'answers'],
    num_rows: 22615
})
{'question': 'why is crop rotation important in farming?', 'answers': 'This helps to prevent soil erosion and depletion, and can also help to control pests and diseases'}


In [6]:
import pandas as pd

df = raw.to_pandas()
print(df.shape)
print(df.isna().sum())
df.sample(5, random_state=SEED)

(22615, 2)
question    0
answers     0
dtype: int64


,question,answers
9334,name any one example of forest pest.,bark beetles
20612,How often can i sprinkle liquid manure?,"Generally, liquid manure can be applied every ..."
7144,how are plants harvested?,by hand or machine
18763,what are some of the advantages of Band placem...,"By placing the fertilizers in rows, you can sp..."
6492,what is rotenone?,Rotenone is a plant extract found in some spec...


## 5. Dataset Preprocessing (clean, filter, leakage-safe split)

In [7]:
def clean(example):
    return {
        "question": str(example["question"]).strip(),
        "answer": str(example["answers"]).strip(),
    }

cleaned = raw.map(clean)
cleaned = cleaned.filter(
    lambda ex: len(ex["question"]) > 5 and len(ex["answer"]) > 5
               and ex["question"].lower() != ex["answer"].lower()
)
cleaned = cleaned.shuffle(seed=SEED)

TRAIN_N, VAL_N, TEST_N = 4000, 500, 500
train_ds = cleaned.select(range(0, TRAIN_N))
val_ds   = cleaned.select(range(TRAIN_N, TRAIN_N + VAL_N))
test_ds  = cleaned.select(range(TRAIN_N + VAL_N, TRAIN_N + VAL_N + TEST_N))

print(len(train_ds), len(val_ds), len(test_ds))

Map:   0%|          | 0/22615 [00:00<?, ? examples/s]

Filter:   0%|          | 0/22615 [00:00<?, ? examples/s]

4000 500 500


## 6. Prompt / Instruction Formatting

In [8]:
SYSTEM = ("You are an agricultural assistant. Answer the farmer's question "
          "clearly, correctly, and concisely, in plain English.")

def to_train_text(example):
    text = (f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
            f"<|im_start|>user\n{example['question']}<|im_end|>\n"
            f"<|im_start|>assistant\n{example['answer']}<|im_end|>")
    return {"text": text}

def to_prompt_only(question):
    return (f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
            f"<|im_start|>user\n{question}<|im_end|>\n"
            f"<|im_start|>assistant\n")

train_sft = train_ds.map(to_train_text)
val_sft   = val_ds.map(to_train_text)
print(train_sft[0]["text"])

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

<|im_start|>system
You are an agricultural assistant. Answer the farmer's question clearly, correctly, and concisely, in plain English.<|im_end|>
<|im_start|>user
what is seed dormancy<|im_end|>
<|im_start|>assistant
Seed dormancy is the condition in which a seed fails to germinate even under conditions that are normally favorable for germination<|im_end|>


## 7. Base Model

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
)
print("Total params (M):", sum(p.numel() for p in base_model.parameters()) / 1e6)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Total params (M): 1543.714304


## 8. Baseline (before any fine-tuning)

In [10]:
@torch.no_grad()
def generate(model, question, max_new_tokens=200):
    prompt = to_prompt_only(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        repetition_penalty=1.1, pad_token_id=tokenizer.pad_token_id,
    )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

for ex in test_ds.select(range(3)):
    print("Q:", ex["question"])
    print("Base model:", generate(base_model, ex["question"]))
    print("Reference :", ex["answer"])
    print("---")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Q: what do i look when inspect the packaging.
Base model: When inspecting packaging for your crops or products, you should focus on several key aspects to ensure quality and safety:

1. **Labeling**: Check that all labels are clear, complete, and accurate. Labels should include information such as product name, batch number, expiration date, storage instructions, and any relevant health warnings.

2. **Packaging Material**: Ensure that the materials used in packaging (such as plastic, paper, cardboard) are appropriate for the type of product being packaged and meet industry standards for durability, protection from moisture, pests, and contamination.

3. **Condition of Packaging**: Look for signs of damage, tampering, or deterioration. This includes checking for leaks, tears, mold, insect infestation, or other physical defects that could affect the integrity of the package.

4. **Quantity and Weight**: Verify that the quantity and weight of the product match what was expected based on 

## 9. Tokenization Check

In [11]:
lengths = [len(tokenizer(t)["input_ids"]) for t in train_sft.select(range(200))["text"]]
print("mean/median/95th pct token length:", np.mean(lengths), np.median(lengths), np.percentile(lengths, 95))
MAX_SEQ_LEN = 512  # chosen from the 95th percentile check above

mean/median/95th pct token length: 78.725 66.0 145.14999999999995


## 10. LoRA Configuration

In [12]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


## 11. QLoRA Configuration

In [13]:
from transformers import BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

qlora_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
qlora_base = prepare_model_for_kbit_training(qlora_base)

qlora_lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
qlora_model = get_peft_model(qlora_base, qlora_lora_config)
qlora_model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


## 12. Training Configuration + LoRA Training

In [16]:
from trl import SFTConfig, SFTTrainer

lora_sft_config = SFTConfig(
    output_dir="outputs/lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    optim="adamw_torch",
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=False,
    seed=SEED,
)

lora_trainer = SFTTrainer(
    model=lora_model,
    args=lora_sft_config,
    train_dataset=train_sft,
    eval_dataset=val_sft,
    processing_class=tokenizer,
)
lora_trainer.train()
lora_trainer.save_model("outputs/lora")
tokenizer.save_pretrained("outputs/lora")


Step,Training Loss,Validation Loss
50,0.837800,0.880571
100,0.776500,0.845798
150,0.818100,0.798393
200,0.721900,0.757457
250,0.727500,0.706686
300,0.562500,0.671023
350,0.573200,0.637497
400,0.558900,0.603649
450,0.492200,0.582174
500,0.542900,0.561479


('outputs/lora/tokenizer_config.json',
 'outputs/lora/special_tokens_map.json',
 'outputs/lora/chat_template.jinja',
 'outputs/lora/vocab.json',
 'outputs/lora/merges.txt',
 'outputs/lora/added_tokens.json',
 'outputs/lora/tokenizer.json')

## 13. QLoRA Training

In [2]:
# ============================================================
# QLoRA SFT — REPLACED TRAINING CONFIG + TRAINING LOOP
# ============================================================

qlora_sft_config = SFTConfig(
    output_dir="outputs/qlora",

    # -------------------------
    # Training duration
    # -------------------------
    num_train_epochs=5,

    # -------------------------
    # Batch / memory efficiency
    # -------------------------
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    # Effective batch size = 2 × 8 = 16
    # This keeps the effective batch reasonably large
    # without requiring large VRAM.

    # -------------------------
    # Learning rate
    # -------------------------
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    # -------------------------
    # Regularization
    # -------------------------
    weight_decay=0.01,
    max_grad_norm=1.0,

    # -------------------------
    # Precision
    # -------------------------
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),

    # -------------------------
    # Optimizer
    # -------------------------
    optim="paged_adamw_8bit",

    # -------------------------
    # Sequence length
    # -------------------------
    max_length=MAX_SEQ_LEN,
    packing=False,

    # -------------------------
    # Evaluation
    # -------------------------
    eval_strategy="steps",
    eval_steps=25,

    # -------------------------
    # Logging
    # -------------------------
    logging_strategy="steps",
    logging_steps=5,

    # -------------------------
    # Checkpoints
    # -------------------------
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,

    # Keep the checkpoint with the
    # lowest validation loss.
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # -------------------------
    # Dataset
    # -------------------------
    dataset_text_field="text",

    # -------------------------
    # Reproducibility
    # -------------------------
    seed=SEED,

    # -------------------------
    # Disable external logging
    # -------------------------
    report_to="none",

    # -------------------------
    # Memory optimization
    # -------------------------
    gradient_checkpointing=True,

    # Required in many HF causal-LM
    # setups when gradient checkpointing
    # is enabled.
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    # -------------------------
    # Keep unused columns
    # -------------------------
    remove_unused_columns=False,
)


# ============================================================
# CREATE TRAINER
# ============================================================

qlora_trainer = SFTTrainer(
    model=qlora_model,

    args=qlora_sft_config,

    train_dataset=train_sft,
    eval_dataset=val_sft,

    processing_class=tokenizer,
)


# ============================================================
# TRAIN
# ============================================================

print("=" * 60)
print("STARTING QLoRA FINE-TUNING")
print("=" * 60)

print(f"Training examples   : {len(train_sft)}")
print(f"Validation examples : {len(val_sft)}")
print(f"Epochs              : {qlora_sft_config.num_train_epochs}")
print(f"Micro batch size    : {qlora_sft_config.per_device_train_batch_size}")
print(f"Gradient accumulation: {qlora_sft_config.gradient_accumulation_steps}")
print(f"Learning rate       : {qlora_sft_config.learning_rate}")
print(f"Max sequence length : {qlora_sft_config.max_length}")

print("=" * 60)


train_result = qlora_trainer.train()


# ============================================================
# TRAINING SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)

print(f"Training loss : {train_result.training_loss:.4f}")

if torch.cuda.is_available():
    print(
        f"Peak GPU memory: "
        f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
    )


# ============================================================
# FINAL VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("FINAL VALIDATION")
print("=" * 60)

eval_results = qlora_trainer.evaluate()

for key, value in eval_results.items():
    if isinstance(value, (int, float)):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")


# ============================================================
# SAVE BEST MODEL / ADAPTER
# ============================================================

print("\n" + "=" * 60)
print("SAVING MODEL")
print("=" * 60)

qlora_trainer.save_model("outputs/qlora")
tokenizer.save_pretrained("outputs/qlora")

print("Saved to: outputs/qlora")
print("=" * 60)

NameError: name 'SFTConfig' is not defined

## 14. Training Curves

In [3]:
import matplotlib.pyplot as plt

def plot_curves(trainer, title):
    history = trainer.state.log_history
    train_loss = [(h["step"], h["loss"]) for h in history if "loss" in h]
    eval_loss = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]
    plt.figure()
    if train_loss:
        plt.plot(*zip(*train_loss), label="train loss")
    if eval_loss:
        plt.plot(*zip(*eval_loss), label="eval loss")
    plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title(title)
    plt.show()

plot_curves(lora_trainer, "LoRA training curve")
plot_curves(qlora_trainer, "QLoRA training curve")

NameError: name 'lora_trainer' is not defined

## 15. Evaluation (ROUGE-L, BERTScore, Exact Match)

In [ ]:
import evaluate as hf_evaluate

rouge = hf_evaluate.load("rouge")
bertscore = hf_evaluate.load("bertscore")

def evaluate_model(model, dataset):
    preds = [generate(model, ex["question"]) for ex in dataset]
    refs = [ex["answer"] for ex in dataset]
    r = rouge.compute(predictions=preds, references=refs)
    b = bertscore.compute(predictions=preds, references=refs, lang="en")
    return {
        "rouge_l": r["rougeL"],
        "bertscore_f1": sum(b["f1"]) / len(b["f1"]),
        "exact_match_rate": sum(p.strip().lower() == r_.strip().lower() for p, r_ in zip(preds, refs)) / len(preds),
    }

from peft import PeftModel

eval_test = test_ds  # use test_ds.select(range(100)) for a faster pass during iteration

base_scores = evaluate_model(base_model, eval_test)
lora_scores = evaluate_model(lora_model, eval_test)
qlora_scores = evaluate_model(qlora_model, eval_test)

print("Base :", base_scores)
print("LoRA :", lora_scores)
print("QLoRA:", qlora_scores)


## 16. Base vs Fine-Tuned Qualitative Comparison

In [ ]:
rows = []
for ex in test_ds.select(range(10)):
    rows.append({
        "question": ex["question"],
        "base_model": generate(base_model, ex["question"]),
        "lora": generate(lora_model, ex["question"]),
        "qlora": generate(qlora_model, ex["question"]),
        "reference": ex["answer"],
    })
comparison_df = pd.DataFrame(rows)
comparison_df


## 17. LoRA vs QLoRA Resource Comparison

In [ ]:
import subprocess

print("LoRA adapter size:")
print(subprocess.run(["du", "-sh", "outputs/lora"], capture_output=True, text=True).stdout)
print("QLoRA adapter size:")
print(subprocess.run(["du", "-sh", "outputs/qlora"], capture_output=True, text=True).stdout)

# Peak memory per run should be captured live during Sections 12-13 with
# torch.cuda.reset_peak_memory_stats() before training and
# torch.cuda.max_memory_allocated() after — record those numbers in results/fine_tuned_results.md.


## 18. Ablation Experiment: LoRA Rank

In [ ]:
ablation_results = {}
for r in [4, 16, 64]:
    cfg_r = LoraConfig(
        r=r, lora_alpha=r * 2, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM", target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    fresh_base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16, device_map="auto"
    )
    model_r = get_peft_model(fresh_base, cfg_r)

    ablation_config = SFTConfig(
        output_dir=f"outputs/ablation_r{r}",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        eval_strategy="no",
        save_strategy="no",
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
        dataset_text_field="text",
        max_length=MAX_SEQ_LEN,
        packing=False,
        seed=SEED,
    )
    trainer_r = SFTTrainer(model=model_r, args=ablation_config, train_dataset=train_sft, processing_class=tokenizer)
    trainer_r.train()
    ablation_results[r] = evaluate_model(model_r, test_ds.select(range(100)))

print(ablation_results)


## 19. Failure Analysis (Required)

In [ ]:
flagged = []
for ex in test_ds.select(range(50)):
    pred = generate(qlora_model, ex["question"])
    if len(pred) < 15 or len(pred) > 600:
        flagged.append({"question": ex["question"], "prediction": pred, "reference": ex["answer"]})

print(f"{len(flagged)} flagged examples out of 50 for manual review")
for f in flagged[:10]:
    print(f)


## 20. Final Conclusions

Fill this cell in only after Sections 15-19 have produced real numbers.
Answer each research question from the introduction using the measured
ROUGE-L/BERTScore/exact-match values, the memory/time/adapter-size
comparison, the rank-ablation results, and the documented failures.
Do not write conclusions here before the experiment has actually run —
copy the final numbers into `results/baseline_results.md` and
`results/fine_tuned_results.md` at the same time.
